# Demo — Medallion Catalog Governance in AWS Glue
In a **Medallion Lakehouse**, AWS Glue table and column `Parameters` (`medallion_zone`, `steward`, `sensitivity`, `retention_policy`) drive automated Lake Formation policies. If a Silver or Bronze table is missing its steward or has an untagged PHI/CONFIDENTIAL column, downstream access controls cannot protect it.

In this demo, we audit Veridian Health's `veridian_lakehouse` Glue catalog and remediate two representative findings in the **Clinical Telehealth** and **Diagnostics & Labs** domains.

## Step 0: Connect to your Cloud Lab

Run the cell below to configure your AWS Cloud Lab environment and seed the **Veridian Health Medallion Lakehouse** (`veridian_lakehouse` in AWS Glue, S3 `bronze/silver/gold` Parquet prefixes, and AWS Lake Formation enforcement) via `setup_aws.py`.

In [1]:
import os
os.environ["VERIDIAN_LF_LIVE"] = "1"
os.environ["VERIDIAN_AWS_REGION"] = "us-east-1"
# If your Vocareum AWS credentials are not already in the environment, paste them below:
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_SESSION_TOKEN"] = "..."

import boto3
import setup_aws
glue = boto3.client("glue", region_name=os.environ["VERIDIAN_AWS_REGION"])
try:
    glue.get_database(Name="veridian_lakehouse")
    print("Glue database 'veridian_lakehouse' is already seeded and ready.")
except glue.exceptions.EntityNotFoundException:
    setup_aws.main()

Glue database 'veridian_lakehouse' is already seeded and ready.


In [2]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from demo_toolkit import connect_lakehouse, lh_backend

lh = connect_lakehouse()
print(f"Backend: {lh_backend()} -> {type(lh).__name__} (Glue DB: {lh.database}, Region: {lh.region})")

Backend: live -> AwsMedallionLakehouse (Glue DB: veridian_lakehouse, Region: us-east-1)


## 1. Run the Medallion Lakehouse catalog audit

In [3]:
audit = lh.audit_lakehouse_catalog()
print(f"Total governance gaps in {lh.database}: {len(audit)}")
display(audit)

Total governance gaps in veridian_lakehouse: 9


                           table    zone                             domain                  gap_type                         detail
0          silver_patient_master  silver          Patient Identity & Intake           untagged_column                mrn (type TEXT)
1          bronze_patient_intake  bronze          Patient Identity & Intake  missing_retention_policy               retention_policy
2      silver_clinical_encounter  silver                Clinical Telehealth           missing_steward                        steward
3              silver_lab_result  silver                 Diagnostics & Labs       missing_sensitivity                    sensitivity
4  bronze_lab_device_calibration  bronze                 Diagnostics & Labs  missing_retention_policy               retention_policy
5  bronze_lab_device_calibration  bronze                 Diagnostics & Labs           untagged_column   reagent_cost_usd (type REAL)
6           silver_claim_billing  silver  Revenue Cycle & Quality Rep

## 2. Remediate a table-level gap and a column-level gap in AWS Glue

First, inspect `silver_clinical_encounter` — it is missing its `steward` parameter in Glue:

In [4]:
display(audit[audit["table"] == "silver_clinical_encounter"])
lh.set_table_governance("silver_clinical_encounter", steward="clinical-ops@veridian.health")

                       table    zone               domain         gap_type   detail
2  silver_clinical_encounter  silver  Clinical Telehealth  missing_steward  steward


Next, inspect the untagged columns across the lakehouse. In `bronze_lab_device_calibration`, `reagent_cost_usd` holds proprietary vendor pricing and should be classified as `CONFIDENTIAL`:

In [5]:
display(audit[audit["gap_type"] == "untagged_column"])
lh.classify_column("bronze_lab_device_calibration", "reagent_cost_usd", "CONFIDENTIAL")
print("Remaining gaps after fixing 2 findings:", len(lh.audit_lakehouse_catalog()))

                           table    zone                             domain         gap_type                         detail
0          silver_patient_master  silver          Patient Identity & Intake  untagged_column                mrn (type TEXT)
5  bronze_lab_device_calibration  bronze                 Diagnostics & Labs  untagged_column   reagent_cost_usd (type REAL)
8           silver_claim_billing  silver  Revenue Cycle & Quality Reporting  untagged_column  patient_email_ref (type TEXT)
Remaining gaps after fixing 2 findings: 7


## 3. Inspect cross-domain semantic conflicts

Even when every Glue tag is populated, domains can still disagree on keys, HIPAA authorization flags, or clinical units:

In [6]:
lh.semantic_conflicts()

                                 concept                   domain_a                                                  field_a                           domain_b                                                                                   field_b                                                                                                                                                                                           impact
0                   patient identity key  Patient Identity & Intake         silver_patient_master.mpi_id (INTEGER surrogate)  Revenue Cycle & Quality Reporting  silver_claim_billing.patient_email_ref (TEXT — 625/650 [96.15%] email strings, 25 'n/a')  625 of 650 (96.15%) billing claims key patients by mutable email string (plus 25 unresolvable 'n/a' rows) instead of mpi_id; joining Silver Patient Master to Claims requires email resolution.
1   secondary-use research authorization  Patient Identity & Intake                silver_patient_master.hipaa_res